# The frontier of RL: reward hacking, RLHF, and the practical reality

_Reinforcement Learning_

**Everything between a textbook MDP and a working RL system — including the headline lesson that agents game any reward you mis-specify.**

The earlier lessons gave you the machinery &mdash; MDPs (Markov Decision Processes), returns, the
       Bellman equations, TD (Temporal-Difference) learning, policy gradients, deep RL. This capstone is about
       the gap between that machinery and a system that works, and where the field is pushing.

       One idea sits above all the others: the agent optimises the reward you actually wrote, with no
       regard for what you meant. Any gap between the two is a vulnerability the optimiser will find &mdash;
       this is reward hacking (a.k.a. specification gaming). It is not a bug in the algorithm; it is the
       algorithm working too well on the wrong objective. So reward design is the real job, and the math
       below (potential-based shaping) is the one tool that lets you help learning without changing
       what the agent is ultimately trying to do.

---

This notebook builds the two headline tools — potential-based reward shaping and an RLHF reward model — one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — numpy (gridworld) — RLHF reward-model snippet in PyTorch

### Step 1 — Build the sparse-reward gridworld

We use a 5x5 grid where the agent starts top-left and must reach the bottom-right goal. The reward is **sparse**: the agent gets `+1` only on the step that lands on the goal, and `0` everywhere else. Sparse reward is exactly the setting where learning is slow, because the agent has to stumble onto the goal by luck before it ever sees a learning signal.

In [ ]:
# Potential-based reward SHAPING speeds up learning without moving the optimum.
# Pure numpy -- runs anywhere (Colab or local).
import numpy as np
rng = np.random.default_rng(0)

N = 5                                   # 5x5 grid
START, GOAL = (0, 0), (N - 1, N - 1)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right
gamma, alpha, eps = 0.95, 0.5, 0.3

def step(s, a):
    dr, dc = ACTIONS[a]
    r, c = s[0] + dr, s[1] + dc
    r, c = min(max(r, 0), N - 1), min(max(c, 0), N - 1)   # clamp to grid
    s2 = (r, c)
    reward = 1.0 if s2 == GOAL else 0.0     # SPARSE: reward only at the goal
    done = (s2 == GOAL)
    return s2, reward, done

### Step 2 — Add a potential and a shaped Q-learning loop

Potential-based shaping adds an extra reward term `F = gamma*Phi(s2) - Phi(s)`, where the potential `Phi(s)` is the negative Manhattan distance to the goal. Because this term telescopes, the theory guarantees it cannot change which policy is optimal — it only gives the agent dense hints that point it toward the goal. The `train` function runs tabular Q-learning with epsilon-greedy exploration, optionally turning the shaping term on, and records how many steps each episode took (fewer steps = solved faster).

In [ ]:
def potential(s):                            # Phi(s) = -(Manhattan distance to goal)
    return -(abs(s[0] - GOAL[0]) + abs(s[1] - GOAL[1]))

def train(shaped, episodes=400, max_steps=500, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((N, N, 4))
    steps_to_goal = []
    for ep in range(episodes):
        s, done, t = START, False, 0
        while not done and t < max_steps:
            a = rng.integers(4) if rng.random() < eps else int(np.argmax(Q[s[0], s[1]]))
            s2, r, done = step(s, a)
            # Potential-based shaping: r' = r + gamma*Phi(s2) - Phi(s).
            r_train = r + (gamma * potential(s2) - potential(s) if shaped else 0.0)
            target = r_train + (0.0 if done else gamma * np.max(Q[s2[0], s2[1]]))
            Q[s[0], s[1], a] += alpha * (target - Q[s[0], s[1], a])
            s = s2
            t += 1
        steps_to_goal.append(t)             # fewer steps = solved faster
    return Q, steps_to_goal

### Step 3 — Compare sparse vs shaped learning

We train one agent on the raw sparse reward and one with shaping, then check the greedy path length each learned. The key result: **both reach the same optimal 8-step path** — shaping did not change the optimum — but the shaped agent gets there far sooner, while the sparse agent flails against the step cap for ~180 episodes before it ever finds the goal.

In [ ]:
Q_sparse, steps_sparse = train(shaped=False)
Q_shaped, steps_shaped = train(shaped=True)

# Greedy path length under each learned Q (same optimum = 8 steps, the shortest path).
def greedy_len(Q):
    s, t = START, 0
    while s != GOAL and t < 100:
        s, _, _ = step(s, int(np.argmax(Q[s[0], s[1]])))
        t += 1
    return t

print("sparse : mean steps over last 50 eps =", round(np.mean(steps_sparse[-50:]), 1),
      "| greedy path =", greedy_len(Q_sparse))
print("shaped : mean steps over last 50 eps =", round(np.mean(steps_shaped[-50:]), 1),
      "| greedy path =", greedy_len(Q_shaped))
# Typical output -- both reach the SAME optimal 8-step greedy path, but the shaped
# agent gets there far sooner (the sparse agent flails at the step cap for ~180
# episodes before it ever stumbles onto the goal):
# sparse : mean steps over last 50 eps = 11.8 | greedy path = 8
# shaped : mean steps over last 50 eps = 11.1 | greedy path = 8

### Step 4 — Sketch an RLHF reward model

Reward shaping helps when you *have* a reward; RLHF is about the harder case where you cannot hand-write one at all (e.g. "be helpful"). Instead you **learn** a reward model `r_phi` from human preference pairs — given a winner `y_w` and loser `y_l`, the Bradley-Terry loss `-log sigma(r(y_w) - r(y_l))` pushes the winner's score above the loser's. This is only stage 1 of RLHF; stage 2 optimizes the language-model policy against `r_phi` with PPO under a KL leash that keeps it from drifting into reward-hacked gibberish.

In [ ]:
# ---------------------------------------------------------------------------
# RLHF reward model (illustrative PyTorch sketch). Colab: already has torch.
# Fits r_phi to human preferences y_w > y_l via the Bradley-Terry loss
# -log sigma( r(x, y_w) - r(x, y_l) ). This is stage 1 of the RLHF pipeline;
# stage 2 optimizes the LANGUAGE-MODEL policy against r_phi with PPO + a KL leash.
import torch
import torch.nn as nn
import torch.nn.functional as Fnn

reward_model = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Linear(256, 1))
opt = torch.optim.Adam(reward_model.parameters(), lr=1e-4)

def preference_loss(emb_w, emb_l):          # emb_* : embeddings of winner / loser responses
    r_w = reward_model(emb_w).squeeze(-1)   # scalar score for the preferred response
    r_l = reward_model(emb_l).squeeze(-1)   # scalar score for the rejected response
    return -Fnn.logsigmoid(r_w - r_l).mean()  # push r_w above r_l

# Training loop sketch (real data = human-labeled (chosen, rejected) pairs):
# for emb_w, emb_l in preference_dataloader:
#     loss = preference_loss(emb_w, emb_l)
#     opt.zero_grad(); loss.backward(); opt.step()

## Visualize the data & results

_Does potential-based reward shaping actually make learning faster -- and does it reach the SAME optimal policy as the raw sparse reward?_

### Step 1 — Re-run sparse vs shaped, keeping the learning curve

To chart *how fast* each agent learns, we re-build the same gridworld and Q-learning loop but now return the full per-episode step count, then smooth it with a 10-episode moving average so the curve is readable. Everything here matches the reference implementation; we only keep the trajectory instead of the final number.

In [ ]:
import numpy as np

# Self-contained 5x5 gridworld, tabular Q-learning, SPARSE vs POTENTIAL-SHAPED reward.
N = 5
START, GOAL = (0, 0), (N - 1, N - 1)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
gamma, alpha, eps = 0.95, 0.5, 0.3

def step(s, a):
    dr, dc = ACTIONS[a]
    r = min(max(s[0] + dr, 0), N - 1)
    c = min(max(s[1] + dc, 0), N - 1)
    s2 = (r, c)
    return s2, (1.0 if s2 == GOAL else 0.0), (s2 == GOAL)

def potential(s):                      # Phi(s) = -(Manhattan distance to goal)
    return -(abs(s[0] - GOAL[0]) + abs(s[1] - GOAL[1]))

def train(shaped, episodes=400, max_steps=500, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((N, N, 4))
    steps = []
    for _ in range(episodes):
        s, done, t = START, False, 0
        while not done and t < max_steps:
            a = rng.integers(4) if rng.random() < eps else int(np.argmax(Q[s[0], s[1]]))
            s2, r, done = step(s, a)
            r_train = r + (gamma * potential(s2) - potential(s) if shaped else 0.0)
            target = r_train + (0.0 if done else gamma * np.max(Q[s2[0], s2[1]]))
            Q[s[0], s[1], a] += alpha * (target - Q[s[0], s[1], a])
            s, t = s2, t + 1
        steps.append(t)
    return np.array(steps)

def smooth(x, w=10):                    # 10-episode moving average for a readable curve
    return np.array([x[max(0, i - w + 1):i + 1].mean() for i in range(len(x))])

sparse = smooth(train(shaped=False))
shaped = smooth(train(shaped=True))

### Step 2 — Report the two learning curves side by side

We subsample both smoothed curves to a handful of episodes and print them together. Read down the `shaped` column: it drops toward the 8-step optimum within ~12 episodes, while `sparse` sits pinned at the 500-step cap until episode ~200, then collapses to the same optimum — slow start, identical destination.

In [ ]:
# Subsample to <= 60 plotted points for the chart.
idxs = [1,3,5,8,12,16,20,25,30,40,50,65,80,100,130,160,200,260,320,400]
for i in idxs:
    print(f"ep={i:>3}  sparse={sparse[i-1]:6.1f}  shaped={shaped[i-1]:6.1f}")
# ep=  1  sparse= 500.0  shaped=  99.0
# ep=  8  sparse= 500.0  shaped=  21.1
# ep= 12  sparse= 500.0  shaped=  11.2   <- shaped already near the 8-step optimum
# ep=160  sparse= 500.0  shaped=  10.8   <- sparse still hitting the 500-step cap
# ep=200  sparse=  14.7  shaped=  13.4   -> sparse finally found the goal & collapsed
# ep=400  sparse=  14.0  shaped=  10.4   -> SAME 8-step greedy optimum, shaped far sooner

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A gridworld gives reward only at the goal (sparse). You add potential-based shaping with $\Phi(s) = -\,\text{(Manhattan distance from }s\text{ to the goal)}$ and $\gamma=0.95$. For a step that moves from distance $4$ to distance $3$, compute the shaping bonus $F$, and state whether the optimal policy can change.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the potentials: $\Phi(s)=-4$ (distance 4), $\Phi(s')=-3$ (distance 3). — _$\Phi$ is just negative distance, so closer states have higher (less negative) potential._
- Apply $F(s,s')=\gamma\Phi(s')-\Phi(s) = 0.95(-3) - (-4) = -2.85 + 4 = +1.15$. — _Moving one step closer to the goal earns a positive shaping bonus, giving dense guidance._
- Recall the theorem: potential-based shaping changes the return by only the constant $-\Phi(s_0)$, so it cannot change which action is optimal. — _The $-\Phi(s)$ shift is identical across all actions in a state, so the $\arg\max$ is untouched._

**Answer:** $F = +1.15$. The optimal policy cannot change: potential-based shaping shifts every policy's value by the same state-dependent constant $-\Phi(s)$, which is constant across actions, so $\arg\max_a Q$ is preserved. You get faster learning, same optimum.

</details>

**Problem 2.** In RLHF, the policy optimisation maximises $\mathbb{E}_{y\sim\pi_\theta}[r_\phi(x,y)] - \beta\,D_{\mathrm{KL}}(\pi_\theta\,\Vert\,\pi_{\text{ref}})$. What goes wrong if you drop the KL term ($\beta = 0$), and what is $r_\phi$ trained from?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- With $\beta=0$ the only objective is "maximise the reward model's score". — _Nothing anchors the policy to sensible language anymore._
- The reward model $r_\phi$ is imperfect; the optimiser finds inputs that score high but are degenerate (repetitive, off-distribution text) — reward hacking. — _Optimising any imperfect learned reward to the extreme exploits its errors._
- $r_\phi$ is trained from human PREFERENCE comparisons ($y_w \succ y_l$) via the Bradley–Terry loss $-\log\sigma(r_\phi(x,y_w)-r_\phi(x,y_l))$. — _You cannot hand-write a reward for 'be helpful', so you learn it from which response humans preferred._

**Answer:** Without the KL leash ($\beta=0$) the policy drifts far from $\pi_{\text{ref}}$ and exploits the imperfect reward model &mdash; producing high-scoring gibberish (reward hacking). The KL penalty keeps $\pi_\theta$ close to the original model. The reward model $r_\phi$ is trained from human preference comparisons $y_w \succ y_l$ using the Bradley&ndash;Terry / logistic loss.

</details>

**Problem 3.** A product team wants to "use RL" to pick which of several banner ads to show a user, where the ad shown does NOT change what the user will be shown next (each impression is independent). Is full RL the right tool? What is simpler?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check the defining property of RL: does today's action change tomorrow's state? — _Sequentiality (actions affect future state) is what distinguishes RL from one-step problems._
- Here each impression is independent — the action does NOT affect the next state. — _That collapses the long-horizon return to a single immediate reward._
- A one-step decision under uncertainty with no state transitions is a MULTI-ARMED BANDIT. — _Bandit algorithms (e.g. UCB, Thompson sampling) solve exactly this and need far less data than full RL._

**Answer:** No &mdash; full RL is overkill. Because the action does not change future state, this is a bandit problem, not a sequential MDP. Use a bandit algorithm (UCB, Thompson sampling): simpler, more sample-efficient, and more stable. Recognising "this is a bandit / supervised / control problem, not RL" is the practical meta-skill &mdash; reach for full RL only when the problem is genuinely sequential.

</details>